# 配套实践 13-02：动作缓冲、延迟与异步滚动执行

本练习模拟一个高频控制线程和一个较慢的策略线程。策略每隔 8 个控制周期读取 Context，经过不固定延迟后返回 12 步动作块。我们比较阻塞推理、直接异步缓冲和带 Context 版本检查的异步缓冲，并在中途加入外部扰动。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/13-generative-action-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 模拟动作缓冲、异步事件和一维位置更新
import matplotlib.pyplot as plt  # 绘制推理时间线、控制轨迹和缓冲状态
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 策略延迟不是一个固定常数

下面给出 8 次策略请求的延迟。请求间隔固定为 8 个控制周期，但返回时间会变化；当延迟大于请求间隔时，新旧推理甚至会重叠。动作缓冲必须覆盖这些间隙，评测也要关注最长延迟而不只是平均值。

In [ ]:
total_steps = 64  # 设置完整模拟包含六十四个控制周期
request_period = 8  # 设置策略每八个控制周期读取一次 Context
chunk_length = 12  # 设置每次策略返回十二步动作块
latencies = np.array([3, 6, 4, 9, 2, 7, 5, 8])  # 定义八次推理的非固定延迟
request_times = np.arange(0, total_steps, request_period)  # 建立固定周期的 Context 请求时刻
arrival_times = request_times + latencies  # 计算各动作块真正返回控制线程的时刻
fig, axis = plt.subplots(figsize=(10, 4.3))  # 创建策略请求与返回的甘特式时间线
for request_index, (request_time, arrival_time) in enumerate(zip(request_times, arrival_times)):  # 逐个绘制策略推理区间
    axis.barh(request_index, arrival_time - request_time, left=request_time, height=0.55, color="#bfdbfe", edgecolor="#2563eb")  # 用横条显示当前 Context 的推理占用时间
    axis.scatter(request_time, request_index, color="#16a34a", zorder=3, label="Context request" if request_index == 0 else None)  # 标出读取 Context 的开始时刻
    axis.scatter(arrival_time, request_index, color="#ea580c", zorder=3, label="Chunk arrival" if request_index == 0 else None)  # 标出对应动作块返回时刻
    axis.text((request_time + arrival_time) / 2, request_index, f"v{request_index}", ha="center", va="center", fontsize=9)  # 在横条中写出 Context 版本
axis.axvline(28, color="#dc2626", linestyle="--", label="External disturbance")  # 标出后续模拟中的外部扰动时刻
axis.set(title="Policy requests have variable latency and can overlap", xlabel="Control step", ylabel="Request index", yticks=range(len(request_times)))  # 标注请求行与控制时间
axis.legend(loc="lower right")  # 显示请求、到达和扰动图例
axis.grid(axis="x", alpha=0.2)  # 添加时间方向淡网格
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示异步策略时间线

**怎样理解结果：** 绿点是读取 Context 的时刻，橙点是对应动作块到达时刻，横条长度就是端到端推理延迟。版本 v3 的延迟超过一个请求周期，因此 v4 已开始计算时它仍未返回。同步阻塞会让控制在这些蓝色区间等待；异步模式则必须继续消费此前动作缓冲。

## 2. 比较阻塞、直接异步与版本检查

一维机器人要到达位置 1。每个动作块根据请求时刻的快照位置计算固定小步，动作返回时这个快照可能已经过时。第 28 步机器人被向后推 0.25。版本检查方案在返回时比较当前状态和快照，差异过大就丢弃旧块。

In [ ]:
target_position = 1.0  # 设置一维控制任务的目标位置
disturbance_step = 28  # 设置外部扰动发生的控制周期
disturbance_value = -0.25  # 设置扰动把机器人向后推的距离
def simulate_controller(mode):  # 定义支持三种推理与缓冲模式的控制模拟
    position = 0.0  # 初始化机器人位置
    buffer = [] if mode == "blocking" else [0.02] * chunk_length  # 异步模式从一个保守初始动作块开始
    pending_requests = []  # 保存尚未返回的策略请求
    position_trace = [position]  # 保存每个控制周期结束后的机器人位置
    buffer_trace = []  # 保存每个周期执行后的缓冲余量
    event_trace = []  # 保存请求、安装和丢弃事件供后续检查
    discarded_results = 0  # 统计因 Context 过期被丢弃的动作块
    empty_buffer_steps = 0  # 统计异步模式没有动作可执行的周期
    blocked_steps = 0  # 统计同步模式等待推理而保持的周期
    for control_step in range(total_steps):  # 逐个推进高频控制周期
        if control_step % request_period == 0 and (mode != "blocking" or len(pending_requests) == 0):  # 按周期发起异步请求并避免阻塞模式重入
            request_index = (control_step // request_period) % len(latencies)  # 找到当前请求对应的延迟索引
            latency = int(latencies[request_index])  # 读取当前策略推理延迟
            snapshot_position = position  # 保存生成该动作块所依据的 Context 状态
            planned_action = (target_position - snapshot_position) / 20.0  # 根据快照把剩余误差分配成小步动作
            planned_chunk = [planned_action] * chunk_length  # 建立固定长度的候选动作块
            pending_requests.append((control_step + latency, control_step, snapshot_position, planned_chunk, request_index))  # 保存预计到达时刻与 Context 元数据
            event_trace.append((control_step, "request", request_index))  # 记录当前版本请求事件
        arriving_requests = [request for request in pending_requests if request[0] == control_step]  # 找出本周期刚刚完成的策略结果
        pending_requests = [request for request in pending_requests if request[0] != control_step]  # 从待完成队列移除已返回结果
        for arrival_time, request_time, snapshot_position, planned_chunk, request_index in arriving_requests:  # 逐个处理同时返回的动作块
            context_shift = abs(position - snapshot_position)  # 比较当前状态与生成时 Context 快照的差异
            if mode == "versioned" and context_shift > 0.12:  # 检查版本化异步模式是否认为结果过期
                discarded_results += 1  # 累计一次过期结果丢弃
                event_trace.append((control_step, "discard", request_index))  # 记录被丢弃的 Context 版本
            else:  # 处理仍可接受的最新动作块
                buffer = planned_chunk.copy()  # 在明确控制周期边界安装完整新动作块
                event_trace.append((control_step, "install", request_index))  # 记录动作块安装事件
        if mode == "blocking" and len(pending_requests) > 0:  # 判断同步策略是否仍在等待当前结果
            action = 0.0  # 等待期间保持当前位置而不消费旧计划
            blocked_steps += 1  # 累计一个阻塞控制周期
        elif len(buffer) > 0:  # 检查动作缓冲是否仍有可执行命令
            action = buffer.pop(0)  # 从缓冲头部消费当前控制动作
        else:  # 处理异步动作缓冲已经耗尽的情况
            action = 0.0  # 使用保守保持动作作为降级
            empty_buffer_steps += 1  # 累计一次缓冲欠载
        position += action  # 根据当前动作更新机器人位置
        if control_step == disturbance_step:  # 检查是否到达外部扰动时刻
            position += disturbance_value  # 把相同向后位移加入三种控制模式
        position_trace.append(position)  # 保存本周期结束后的实际位置
        buffer_trace.append(len(buffer))  # 保存消费动作后的缓冲余量
    return {"mode": mode, "positions": np.array(position_trace), "buffer": np.array(buffer_trace), "events": event_trace, "discarded": discarded_results, "empty": empty_buffer_steps, "blocked": blocked_steps}  # 返回完整轨迹与运行指标
simulation_results = [simulate_controller("blocking"), simulate_controller("naive"), simulate_controller("versioned")]  # 分别运行阻塞、直接异步和版本化异步控制
display_names = ["Blocking", "Async naive", "Async versioned"]  # 定义三种方案的图例名称
display_colors = ["#ea580c", "#94a3b8", "#2563eb"]  # 为三种控制方案分配颜色
fig, axis = plt.subplots(figsize=(10, 4.2))  # 创建扰动下的位置轨迹对比图
for result, name, color in zip(simulation_results, display_names, display_colors):  # 依次绘制三种执行模式
    axis.plot(result["positions"], label=name, color=color, linewidth=2.4)  # 显示机器人位置随控制周期的变化
axis.axhline(target_position, color="#16a34a", linestyle="--", label="Target")  # 标出目标位置
axis.axvline(disturbance_step + 1, color="#dc2626", linestyle=":", label="Disturbance")  # 标出扰动进入状态后的时刻
axis.set(title="A buffer keeps control moving while policy inference runs", xlabel="Control step", ylabel="Position")  # 标注缓冲对控制轨迹的影响
axis.legend()  # 显示三种模式、目标与扰动图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察最终误差
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示延迟和扰动下的三种控制结果

**怎样理解结果：** 阻塞方案在每次推理期间保持不动，六十四个周期后仍明显落后目标。两种异步方案在推理期间继续执行缓冲，因此能持续前进，并在扰动后的新动作块到达后重新接近目标。直接异步会无条件安装迟到计划；版本化方案至少检查该计划依据的状态是否已经过旧。

## 3. 轨迹之外还要检查缓冲健康度

最终位置相近不代表运行过程相同。下面同时显示缓冲余量，并汇总最终误差、阻塞周期、缓冲欠载和丢弃过期结果的次数。

In [ ]:
fig, axis = plt.subplots(figsize=(10, 3.8))  # 创建独立的动作缓冲余量图
for result, name, color in zip(simulation_results, display_names, display_colors):  # 依次绘制三种模式的动作缓冲余量
    axis.step(np.arange(total_steps), result["buffer"], where="post", label=name, color=color, linewidth=2)  # 显示每个控制周期结束后的剩余动作数
axis.set(title="Action buffer occupancy", xlabel="Control step", ylabel="Remaining actions")  # 标注缓冲余量与控制时间
axis.legend()  # 显示三种控制模式图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察缓冲何时耗尽
fig.tight_layout()  # 调整缓冲图像边距
plt.show()  # 单独显示动作缓冲余量避免与不同量纲指标混合
metric_names = ["Final error", "Blocked steps", "Empty buffer", "Discarded stale"]  # 定义四项端到端运行指标
metric_values = []  # 准备保存三种模式的指标行
for result in simulation_results:  # 逐个汇总控制模拟结果
    final_error = abs(result["positions"][-1] - target_position)  # 计算最终位置到目标的绝对误差
    metric_values.append([final_error, result["blocked"], result["empty"], result["discarded"]])  # 保存误差、阻塞、欠载和丢弃计数
metric_values = np.array(metric_values)  # 把指标列表转换为三行四列数组
fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.4))  # 为不同量纲的四个健康指标分别创建坐标轴
for metric_index, axis in enumerate(axes):  # 依次绘制最终误差、阻塞、欠载和丢弃次数
    bars = axis.bar(display_names, metric_values[:, metric_index], color=display_colors)  # 在当前指标中比较三种控制模式
    axis.set_title(metric_names[metric_index])  # 标注当前子图的端到端指标
    axis.tick_params(axis="x", rotation=35)  # 旋转模式名称避免文字重叠
    for bar, value in zip(bars, metric_values[:, metric_index]):  # 逐个读取当前指标的三根柱子
        axis.text(bar.get_x() + bar.get_width() / 2, value + max(metric_values[:, metric_index].max() * 0.04, 0.02), f"{value:.2f}", ha="center", fontsize=8)  # 在柱子上方显示具体结果
fig.suptitle("End-to-end control health metrics use separate scales")  # 强调不同量纲应分开阅读
fig.tight_layout()  # 调整四幅指标图间距
plt.show()  # 显示可读的控制健康度指标

**怎样理解结果：** 阻塞方案没有缓冲欠载统计，却付出了大量 blocked steps 和较大最终误差；异步缓冲会周期性被新块补满，也可能在长延迟时降到零。版本化方案丢弃了一次状态差异过大的旧计划，因此产生更多保持周期，但避免把明显过期动作强行安装。

**本练习的结论：** 生成式 Action Model 的实时性不能只用单次平均推理时间描述。动作缓冲余量、尾延迟、Context 版本、过期丢弃和降级动作都会改变闭环结果，必须与模型质量一起测试。